# Procurement Cost & Supplier Risk Analytics\nSynthetic-data portfolio project covering PPV, OTD, supplier risk and negotiation scenarios.

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\ndf = pd.read_csv('../data/procurement_data.csv', parse_dates=['po_date'])\ndf.head()

## Executive KPIs

In [ ]:
print(f'Total Spend: €{df.actual_spend.sum():,.0f}')\nprint(f'Total PPV: €{df.ppv.sum():,.0f}')\nprint(f'OTD: {df.on_time.mean()*100:.1f}%')

## Supplier Risk Score\n60% normalized unfavorable PPV + 40% late-delivery rate.

In [ ]:
s = df.groupby('supplier').agg(total_spend=('actual_spend','sum'), total_ppv=('ppv','sum'), otd_pct=('on_time','mean')).reset_index()\ns['otd_pct'] *= 100\nu=s.total_ppv.clip(lower=0)\ns['ppv_risk_norm']=100*(u-u.min())/(u.max()-u.min())\ns['risk_score']=(.6*s.ppv_risk_norm+.4*(100-s.otd_pct)).round(1)\ns.sort_values('risk_score',ascending=False).head(10)

## Top PPV Drivers

In [ ]:
top=s.nlargest(10,'total_ppv')\ntop.plot.bar(x='supplier',y='total_ppv',legend=False,title='Top Suppliers by Unfavorable PPV')\nplt.ylabel('PPV (€)')\nplt.tight_layout(); plt.show()

## Supplier Risk Matrix

In [ ]:
plt.figure(figsize=(8,5))\nplt.scatter(s.total_ppv,s.otd_pct,s=s.total_spend/5000,alpha=.65)\nfor _,r in s.nlargest(5,'risk_score').iterrows(): plt.annotate(r.supplier,(r.total_ppv,r.otd_pct))\nplt.xlabel('Total PPV (€)'); plt.ylabel('OTD (%)'); plt.title('Supplier Cost vs Delivery Risk')\nplt.tight_layout(); plt.show()

## What-if Savings Simulator

In [ ]:
scope=df[df.ppv>0]\nfor pct in [.03,.05,.10]:\n    print(f'{pct:.0%} scenario: €{(scope.actual_spend*pct).sum():,.0f}')

## Business Recommendations\nPrioritize suppliers combining unfavorable PPV, meaningful spend exposure and weak OTD. Focus negotiations at supplier-material level and monitor delivery performance after sourcing actions.